# Olist table formats, manually: read every operation

Run this notebook in a **fresh kernel**, one cell at a time. It is self-contained: it imports installed libraries and defines its own small utilities; it does not import the lab runner, adapters or project helpers, and does not invoke project scripts.

The lesson is visible SQL and DataFrame writes, followed immediately by actual metadata. First create Iceberg, Delta and Hudi tables, inspecting each; then insert, update and delete in each format. Defaults give **3 → 5 → 5 → 4**. The separate automated notebook omits update and gives **3 → 5 → 4**.

Source rows and nullable lifecycle fields come from Olist. Reserved source rows simulate inserts; shifting one estimated delivery date is a synthetic update; deleting selected keys is a controlled demonstration. UTC is a lab parsing convention, not verified source timezone provenance. This is sequential correctness on the pinned local stack, not a benchmark or cloud interoperability test.

Each top-to-bottom run gets a new `olist/manual/` storage prefix and its own ignored evidence directory. It never resets another run, changes the automated current-run selector, or writes the source CSV. Do not run two lab writers simultaneously. If a cell fails, restart the kernel and start a fresh run rather than replaying a partly committed mutation.


## 1. Configuration, paths and native schema

Edit the four sampling values below. Initial rows must be positive, insert/delete counts nonnegative, and deletions cannot exceed initial plus inserted rows. The single update always targets the first initial order and adds one day to its required estimated delivery date. Default deletion selects sample rank 2, leaving the updated rank 1 visible.

The Docker runtime mounts the downloaded CSV at `/source/olist_orders_dataset.csv` **read-only**. Only local S3 at the Compose service `s3` is used. The generated evidence contains source-derived data and stays outside Git.


In [ ]:
import csv
import hashlib
import json
import os
import re
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path
from IPython.display import display, JSON
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

CONFIG = {"initial_rows": 3, "insert_rows": 2, "delete_rows": 1,
          "sample_seed": "open-table-formats-lab-v1"}
for key in ["initial_rows", "insert_rows", "delete_rows"]:
    assert type(CONFIG[key]) is int and CONFIG[key] >= 0, key
assert CONFIG["initial_rows"] > 0
TOTAL = CONFIG["initial_rows"] + CONFIG["insert_rows"]
assert CONFIG["delete_rows"] <= TOTAL
assert isinstance(CONFIG["sample_seed"], str) and CONFIG["sample_seed"].strip()

CSV_PATH = Path(os.environ.get("OLIST_CSV_PATH", "/source/olist_orders_dataset.csv"))
assert CSV_PATH.is_file(), "Download Olist and set OLIST_SOURCE_CSV before starting Docker"
assert os.environ.get("S3_ENDPOINT", "http://s3:4566") == "http://s3:4566", "Local lab only"
assert os.environ.get("TZ") == "UTC", "Use the supplied UTC Docker runtime"
SOURCE_SHA = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
RUN_ID = "manual_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_PREFIX = "s3a://lakehouse-lab/olist/manual/" + RUN_ID
EVIDENCE = Path("/workspace/evidence/olist/manual") / RUN_ID
EVIDENCE.mkdir(parents=True, exist_ok=False)

COLUMNS = ["order_id", "customer_id", "order_status", "order_purchase_timestamp",
           "order_approved_at", "order_delivered_carrier_date",
           "order_delivered_customer_date", "order_estimated_delivery_date"]
TIMESTAMPS = COLUMNS[3:]
REQUIRED = COLUMNS[:4] + [COLUMNS[-1]]
SCHEMA = StructType([StructField(name, TimestampType() if name in TIMESTAMPS else StringType(),
                                name not in REQUIRED) for name in COLUMNS])
ICEBERG_TABLE = "manual_iceberg.demo.orders"
ICEBERG_ROOT = RUN_PREFIX + "/iceberg/demo/orders"
DELTA_ROOT = RUN_PREFIX + "/delta/orders"
DELTA_TABLE = "delta.`" + DELTA_ROOT + "`"
HUDI_ROOT = RUN_PREFIX + "/hudi/orders"
progress = {name: "new" for name in ["iceberg", "delta", "hudi"]}
checks, session_checks, metadata_reads = [], [], []
print({"run_id": RUN_ID, "expected_counts": [CONFIG["initial_rows"], TOTAL, TOTAL, TOTAL-CONFIG["delete_rows"]]})


## 2. Spark setup: separate contexts for distinct catalog settings

The installed stack is Spark 3.5.7 / Scala 2.12, Iceberg 1.11.0, Delta 3.3.2 and Hudi 1.2.0. Delta uses `DeltaCatalog` for `spark_catalog`; Hudi SQL uses `HoodieCatalog`. A single `spark_catalog` setting cannot have both values. Extensions are chosen when a session is built; a new session view is not a way to replace those startup settings.

The short factory below shows every configuration. We stop each Spark context before opening the next. Iceberg uses its own Hadoop catalog under this run's prefix; **no Glue or cloud catalog** is used. Local S3 settings and dummy credentials are visible. Hudi's Kryo registrator is configured only in its own context. The notebook prints and records the actual active extension/catalog for each session.


In [ ]:
FORMAT_SETTINGS = {
    "iceberg": {
        "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
        "spark.sql.catalog.manual_iceberg": "org.apache.iceberg.spark.SparkCatalog",
        "spark.sql.catalog.manual_iceberg.type": "hadoop",
        "spark.sql.catalog.manual_iceberg.warehouse": RUN_PREFIX + "/iceberg",
    },
    "delta": {
        "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
        "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    },
    "hudi": {
        "spark.sql.extensions": "org.apache.spark.sql.hudi.HoodieSparkSessionExtension",
        "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.hudi.catalog.HoodieCatalog",
        "spark.kryo.registrator": "org.apache.spark.HoodieSparkKryoRegistrar",
    },
}

def open_spark(format_name):
    assert SparkSession.getActiveSession() is None, "Stop this kernel's prior Spark session first"
    builder = (SparkSession.builder.master("local[2]").appName(RUN_ID + "-" + format_name)
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.shuffle.partitions", "2")
        .config("spark.default.parallelism", "2")
        .config("spark.sql.warehouse.dir", "/tmp/" + RUN_ID + "/" + format_name)
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.endpoint", "http://s3:4566")
        .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.hadoop.fs.s3a.access.key", "test")
        .config("spark.hadoop.fs.s3a.secret.key", "test")
        .config("spark.hadoop.fs.s3a.change.detection.mode", "none"))
    for key, value in FORMAT_SETTINGS[format_name].items():
        builder = builder.config(key, value)
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    actual = {"format": format_name, "spark_version": spark.version,
              "extensions": spark.conf.get("spark.sql.extensions"),
              "session_catalog": spark.conf.get("spark.sql.catalog.spark_catalog", "builtin")}
    assert actual["spark_version"] == "3.5.7"
    assert actual["extensions"] == FORMAT_SETTINGS[format_name]["spark.sql.extensions"]
    if format_name in ["delta", "hudi"]:
        assert actual["session_catalog"] == FORMAT_SETTINGS[format_name]["spark.sql.catalog.spark_catalog"]
    session_checks.append(actual)
    print(actual)
    return spark


## 3. Read, parse and sample the source directly

The code checks the native header, required values, string keys, uniqueness and timestamp parsing. Empty optional lifecycle timestamps remain null. Spark uses UTC to interpret the source's offset-free timestamps; that convention does not establish the source's original timezone.

Rank keys by `SHA256(sample_seed + ":" + order_id)`, then by key. Take the first initial rows and reserve the next rows for insertion. Whole-source validation happens before any table write. The small selected rows are kept in Python so we can recreate a DataFrame in each independent Spark context.


In [ ]:
with CSV_PATH.open(newline="", encoding="utf-8") as source:
    assert next(csv.reader(source)) == COLUMNS, "Unexpected CSV header"
spark = open_spark("iceberg")
strings = StructType([StructField(name, StringType(), True) for name in COLUMNS])
raw = spark.read.option("header", "true").option("mode", "FAILFAST").schema(strings).csv(str(CSV_PATH))
SOURCE_ROWS = raw.count()
assert TOTAL <= SOURCE_ROWS, "Not enough source rows for the configured sample"
for name in REQUIRED:
    assert raw.filter(F.col(name).isNull() | (F.trim(F.col(name)) == "")).limit(1).count() == 0, name
assert raw.filter(~F.col("order_id").rlike("^[0-9a-fA-F]{32}$")).limit(1).count() == 0
assert raw.groupBy("order_id").count().filter("count > 1").limit(1).count() == 0
parsed = raw.select(*[F.to_timestamp(name, "yyyy-MM-dd HH:mm:ss").alias(name)
                     if name in TIMESTAMPS else F.col(name) for name in COLUMNS])
for name in TIMESTAMPS:
    assert raw.filter(F.col(name).isNotNull() & (F.trim(F.col(name)) != "") &
                      F.to_timestamp(name, "yyyy-MM-dd HH:mm:ss").isNull()).limit(1).count() == 0, name
selected = (parsed.orderBy(F.sha2(F.concat(F.lit(CONFIG["sample_seed"] + ":"), F.col("order_id")), 256), "order_id")
                  .limit(TOTAL).collect())
selected_rows = [row.asDict() for row in selected]
initial_rows = selected_rows[:CONFIG["initial_rows"]]
insert_rows = selected_rows[CONFIG["initial_rows"]:]
UPDATE_KEY = initial_rows[0]["order_id"]
DELETE_KEYS = [selected_rows[(index + 1) % TOTAL]["order_id"] for index in range(CONFIG["delete_rows"])]
DELETE_SQL = ",".join("'" + key + "'" for key in DELETE_KEYS)
print({"source_rows": SOURCE_ROWS, "source_sha256": SOURCE_SHA,
       "selected_rows": len(selected_rows), "optional_null_counts":
       {name: parsed.filter(F.col(name).isNull()).count() for name in TIMESTAMPS if name not in REQUIRED}})
spark.createDataFrame(selected_rows, SCHEMA).show(truncate=False)


## 4. Independent expected values and three transparent utilities

Compute expected rows from the selected source, before reading any table. The update is explicitly invented for the lesson. Insertion/deletion counts can be zero; those cells then record a no-op instead of inventing a commit.

The remaining utilities only (1) list objects under this run's prefix, (2) read actual bounded metadata text or decode Avro through the installed Java library, and (3) compare every native business value and save a local snapshot. They do not hide any table operation. Metadata previews are limited to 1 MB and 20 Avro records; each preview reports truncation. Raw business rows and metadata remain in ignored evidence.


In [ ]:
expected = {"create": [row.copy() for row in initial_rows],
            "insert": [row.copy() for row in selected_rows]}
expected["update"] = [row.copy() for row in selected_rows]
for row in expected["update"]:
    if row["order_id"] == UPDATE_KEY:
        row["order_estimated_delivery_date"] += timedelta(days=1)
expected["delete"] = [row.copy() for row in expected["update"] if row["order_id"] not in DELETE_KEYS]

def s3_files(spark, uri):
    assert uri.startswith(RUN_PREFIX + "/"), "Only this manual run may be inspected"
    jvm = spark.sparkContext._jvm
    path = jvm.org.apache.hadoop.fs.Path(uri)
    fs = path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
    if not fs.exists(path):
        return []
    iterator, files = fs.listFiles(path, True), []
    while iterator.hasNext():
        item = iterator.next()
        files.append({"path": item.getPath().toString(), "bytes": item.getLen()})
    return sorted(files, key=lambda item: item["path"])

def read_metadata(spark, uri):
    assert uri.startswith(RUN_PREFIX + "/"), "Only this manual run may be inspected"
    jvm, conf = spark.sparkContext._jvm, spark.sparkContext._jsc.hadoopConfiguration()
    path = jvm.org.apache.hadoop.fs.Path(uri)
    fs = path.getFileSystem(conf)
    assert fs.getFileStatus(path).getLen() <= 1_000_000, "Metadata preview exceeds 1 MB"
    stream, output = fs.open(path), jvm.java.io.ByteArrayOutputStream()
    try:
        jvm.org.apache.hadoop.io.IOUtils.copyBytes(stream, output, conf, False)
        raw_bytes = bytes(output.toByteArray())
    finally:
        stream.close()
    if raw_bytes.startswith(b"Obj\x01"):
        reader = jvm.org.apache.avro.file.DataFileStream(fs.open(path), jvm.org.apache.avro.generic.GenericDatumReader())
        records = []
        try:
            while reader.hasNext() and len(records) < 20:
                records.append(json.loads(reader.next().toString()))
            result = {"encoding": "decoded Avro container", "records": records, "truncated": reader.hasNext()}
        finally:
            reader.close()
    else:
        result = {"encoding": "raw UTF-8 metadata text", "text": raw_bytes.decode("utf-8"), "truncated": False}
    metadata_reads.append({"path": uri, **result})
    return result

def verify(stage, format_name, frame, inventory):
    assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA, "Source changed"
    actual = [row.asDict() for row in frame.select(*COLUMNS).orderBy("order_id").collect()]
    wanted = sorted(expected[stage], key=lambda row: row["order_id"])
    assert actual == wanted, format_name + " " + stage + " business values differ"
    result = {"run_id": RUN_ID, "format": format_name, "stage": stage, "status": "PASS",
              "rows": len(actual), "business_rows": actual, "configuration": CONFIG,
              "source_sha256": SOURCE_SHA, "inventory": inventory, "metadata": list(metadata_reads)}
    (EVIDENCE / (format_name + "-" + stage + ".json")).write_text(json.dumps(result, indent=2, default=str))
    checks.append({"format": format_name, "stage": stage, "rows": len(actual), "status": "PASS"})
    progress[format_name] = stage
    frame.select(*COLUMNS).orderBy("order_id").show(min(TOTAL, 20), truncate=False)
    print({"format": format_name, "stage": stage, "rows": len(actual), "status": "PASS"})


## 5. Create Iceberg, then inspect its table metadata

The first Spark context is already open for Iceberg. A fresh namespace and table are registered in this run's Hadoop catalog. Copy-on-write update/delete settings are explicit. Read the snapshot and manifest metadata tables, actual JSON, and decoded manifest-list Avro. Older snapshots can retain physical files that are no longer current.


In [ ]:
assert progress["iceberg"] == "new" and not s3_files(spark, ICEBERG_ROOT)
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
spark.createDataFrame(initial_rows, SCHEMA).createOrReplaceTempView("initial_orders")
spark.sql("CREATE NAMESPACE manual_iceberg.demo")
spark.sql(f"""CREATE TABLE {ICEBERG_TABLE} USING iceberg
    TBLPROPERTIES ('format-version'='2', 'write.update.mode'='copy-on-write',
                   'write.delete.mode'='copy-on-write') AS SELECT * FROM initial_orders""")


In [ ]:
metadata_reads.clear()
spark.table(ICEBERG_TABLE + ".snapshots").select("snapshot_id", "operation", "manifest_list").show(truncate=False)
spark.table(ICEBERG_TABLE + ".manifests").show(truncate=False)
spark.table(ICEBERG_TABLE + ".files").select("file_path", "record_count").show(truncate=False)
inventory = s3_files(spark, ICEBERG_ROOT)
metadata_paths = [item["path"] for item in inventory if item["path"].endswith(".metadata.json")]
latest = max(metadata_paths, key=lambda path: int(re.search(r"v(\d+)\.metadata\.json$", path).group(1)))
raw_json = read_metadata(spark, latest)
display(JSON(json.loads(raw_json["text"])))
manifest_list = json.loads(raw_json["text"])["snapshots"][-1]["manifest-list"]
display(JSON(read_metadata(spark, manifest_list)))

verify("create", "iceberg", spark.table(ICEBERG_TABLE), inventory)
spark.stop()


## 6. Create Delta, then inspect its transaction log

Open a new context with only Delta's extension/catalog. `errorifexists` refuses to overwrite an existing table. The displayed JSON is the actual latest `_delta_log` commit, not a made-up action summary. A checkpoint is not assumed to exist in this small run.


In [ ]:
spark = open_spark("delta")
assert progress["delta"] == "new" and not s3_files(spark, DELTA_ROOT)
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
spark.createDataFrame(initial_rows, SCHEMA).write.format("delta").mode("errorifexists").save(DELTA_ROOT)


In [ ]:
metadata_reads.clear()
spark.sql("DESCRIBE HISTORY " + DELTA_TABLE).select("version", "operation", "operationMetrics").show(truncate=False)
inventory = s3_files(spark, DELTA_ROOT)
log_paths = [item["path"] for item in inventory if "/_delta_log/" in item["path"] and item["path"].endswith(".json")]
latest = max(log_paths)
raw_log = read_metadata(spark, latest)
print("Actual commit JSON:", latest)
print(raw_log["text"])

verify("create", "delta", spark.read.format("delta").load(DELTA_ROOT), inventory)
spark.stop()


## 7. Create Hudi Copy-on-Write, then inspect its timeline

Open Hudi's own context. Keep `order_id` as the record key; use a nonpartitioned Copy-on-Write table and commit-time ordering. Optional lifecycle timestamps are not used as ordering fields. The small local parallelism settings are teaching choices, not performance comparisons.

The business timeline and auxiliary metadata table are separate. Auxiliary log files do not make the business table Merge-on-Read. We show actual properties and decode the latest completed business commit; no JSON encoding is assumed when the file is Avro.


In [ ]:
spark = open_spark("hudi")
assert progress["hudi"] == "new" and not s3_files(spark, HUDI_ROOT)
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
HUDI_OPTIONS = {
    "hoodie.table.name": "manual_orders",
    "hoodie.datasource.write.recordkey.field": "order_id",
    "hoodie.datasource.write.partitionpath.field": "",
    "hoodie.datasource.write.keygenerator.class": "org.apache.hudi.keygen.NonpartitionedKeyGenerator",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.record.merge.mode": "COMMIT_TIME_ORDERING",
    "hoodie.clean.automatic": "false",
    "hoodie.metadata.enable": "true",
    "hoodie.bulk_insert.shuffle.parallelism": "2",
    "hoodie.insert.shuffle.parallelism": "2",
    "hoodie.upsert.shuffle.parallelism": "2",
    "hoodie.delete.shuffle.parallelism": "2",
}
(spark.createDataFrame(initial_rows, SCHEMA).write.format("hudi").options(**HUDI_OPTIONS)
      .option("hoodie.datasource.write.operation", "bulk_insert").mode("errorifexists").save(HUDI_ROOT))


In [ ]:
metadata_reads.clear()
inventory = s3_files(spark, HUDI_ROOT)
main = [item for item in inventory if "/.hoodie/metadata/" not in item["path"]]
auxiliary = [item for item in inventory if "/.hoodie/metadata/" in item["path"]]
timeline = [item["path"] for item in main if "/.hoodie/timeline/" in item["path"]]
display(JSON({"main_timeline": timeline, "business_log_objects": sum(".log." in item["path"] for item in main),
              "auxiliary_metadata_log_objects": sum(".log." in item["path"] for item in auxiliary)}))
display(JSON(read_metadata(spark, HUDI_ROOT + "/.hoodie/hoodie.properties")))
completed = [path for path in timeline if path.endswith(".commit")]
assert completed, "No completed business commit found"
display(JSON(read_metadata(spark, max(completed))))

verify("create", "hudi", spark.read.format("hudi").load(HUDI_ROOT), inventory)
spark.stop()


## 8. Insert the reserved orders

Insert the next distinct source ranks into all three tables. This simulates arrival; it is not the dataset's real event chronology. Default counts rise from 3 to 5. With zero inserts, skip the write and inspect the unchanged metadata.

Each format gets its own newly configured Spark context. The writes and metadata queries are directly below.


### Iceberg insert

Read the operation, run it once, then inspect the actual metadata in the next cell.


In [ ]:
spark = open_spark("iceberg")
assert progress["iceberg"] == "create", "Run cells once in order"
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
if CONFIG["insert_rows"]:
    spark.createDataFrame(insert_rows, SCHEMA).createOrReplaceTempView("incoming_orders")
    spark.sql(f"INSERT INTO {ICEBERG_TABLE} SELECT * FROM incoming_orders")
else:
    print("No-op: zero inserts; no commit requested")


In [ ]:
metadata_reads.clear()
spark.table(ICEBERG_TABLE + ".snapshots").select("snapshot_id", "operation", "manifest_list").show(truncate=False)
spark.table(ICEBERG_TABLE + ".manifests").show(truncate=False)
spark.table(ICEBERG_TABLE + ".files").select("file_path", "record_count").show(truncate=False)
inventory = s3_files(spark, ICEBERG_ROOT)
metadata_paths = [item["path"] for item in inventory if item["path"].endswith(".metadata.json")]
latest = max(metadata_paths, key=lambda path: int(re.search(r"v(\d+)\.metadata\.json$", path).group(1)))
raw_json = read_metadata(spark, latest)
display(JSON(json.loads(raw_json["text"])))
manifest_list = json.loads(raw_json["text"])["snapshots"][-1]["manifest-list"]
display(JSON(read_metadata(spark, manifest_list)))

verify("insert", "iceberg", spark.table(ICEBERG_TABLE), inventory)
spark.stop()


### Delta insert

Read the operation, run it once, then inspect the actual metadata in the next cell.


In [ ]:
spark = open_spark("delta")
assert progress["delta"] == "create", "Run cells once in order"
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
if CONFIG["insert_rows"]:
    spark.createDataFrame(insert_rows, SCHEMA).write.format("delta").mode("append").save(DELTA_ROOT)
else:
    print("No-op: zero inserts; no commit requested")


In [ ]:
metadata_reads.clear()
spark.sql("DESCRIBE HISTORY " + DELTA_TABLE).select("version", "operation", "operationMetrics").show(truncate=False)
inventory = s3_files(spark, DELTA_ROOT)
log_paths = [item["path"] for item in inventory if "/_delta_log/" in item["path"] and item["path"].endswith(".json")]
latest = max(log_paths)
raw_log = read_metadata(spark, latest)
print("Actual commit JSON:", latest)
print(raw_log["text"])

verify("insert", "delta", spark.read.format("delta").load(DELTA_ROOT), inventory)
spark.stop()


### Hudi insert

Read the operation, run it once, then inspect the actual metadata in the next cell.


In [ ]:
spark = open_spark("hudi")
assert progress["hudi"] == "create", "Run cells once in order"
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
if CONFIG["insert_rows"]:
    (spark.createDataFrame(insert_rows, SCHEMA).write.format("hudi").options(**HUDI_OPTIONS)
          .option("hoodie.datasource.write.operation", "insert").mode("append").save(HUDI_ROOT))
else:
    print("No-op: zero inserts; no commit requested")


In [ ]:
metadata_reads.clear()
inventory = s3_files(spark, HUDI_ROOT)
main = [item for item in inventory if "/.hoodie/metadata/" not in item["path"]]
auxiliary = [item for item in inventory if "/.hoodie/metadata/" in item["path"]]
timeline = [item["path"] for item in main if "/.hoodie/timeline/" in item["path"]]
display(JSON({"main_timeline": timeline, "business_log_objects": sum(".log." in item["path"] for item in main),
              "auxiliary_metadata_log_objects": sum(".log." in item["path"] for item in auxiliary)}))
display(JSON(read_metadata(spark, HUDI_ROOT + "/.hoodie/hoodie.properties")))
completed = [path for path in timeline if path.endswith(".commit")]
assert completed, "No completed business commit found"
display(JSON(read_metadata(spark, max(completed))))

verify("insert", "hudi", spark.read.format("hudi").load(HUDI_ROOT), inventory)
spark.stop()


## 9. Update one estimated delivery date

Add one day to the first initial order's estimated delivery date. This value is explicitly synthetic. The row count stays 5 by default. Observe the new commit/snapshot and compare every business value.

Each format gets its own newly configured Spark context. The writes and metadata queries are directly below.


### Iceberg update

Read the operation, run it once, then inspect the actual metadata in the next cell.


In [ ]:
spark = open_spark("iceberg")
assert progress["iceberg"] == "insert", "Run cells once in order"
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
spark.sql(f"UPDATE {ICEBERG_TABLE} SET order_estimated_delivery_date = order_estimated_delivery_date + INTERVAL 1 DAY WHERE order_id = '{UPDATE_KEY}'")


In [ ]:
metadata_reads.clear()
spark.table(ICEBERG_TABLE + ".snapshots").select("snapshot_id", "operation", "manifest_list").show(truncate=False)
spark.table(ICEBERG_TABLE + ".manifests").show(truncate=False)
spark.table(ICEBERG_TABLE + ".files").select("file_path", "record_count").show(truncate=False)
inventory = s3_files(spark, ICEBERG_ROOT)
metadata_paths = [item["path"] for item in inventory if item["path"].endswith(".metadata.json")]
latest = max(metadata_paths, key=lambda path: int(re.search(r"v(\d+)\.metadata\.json$", path).group(1)))
raw_json = read_metadata(spark, latest)
display(JSON(json.loads(raw_json["text"])))
manifest_list = json.loads(raw_json["text"])["snapshots"][-1]["manifest-list"]
display(JSON(read_metadata(spark, manifest_list)))

verify("update", "iceberg", spark.table(ICEBERG_TABLE), inventory)
spark.stop()


### Delta update

Read the operation, run it once, then inspect the actual metadata in the next cell.


In [ ]:
spark = open_spark("delta")
assert progress["delta"] == "insert", "Run cells once in order"
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
spark.sql(f"UPDATE {DELTA_TABLE} SET order_estimated_delivery_date = order_estimated_delivery_date + INTERVAL 1 DAY WHERE order_id = '{UPDATE_KEY}'")


In [ ]:
metadata_reads.clear()
spark.sql("DESCRIBE HISTORY " + DELTA_TABLE).select("version", "operation", "operationMetrics").show(truncate=False)
inventory = s3_files(spark, DELTA_ROOT)
log_paths = [item["path"] for item in inventory if "/_delta_log/" in item["path"] and item["path"].endswith(".json")]
latest = max(log_paths)
raw_log = read_metadata(spark, latest)
print("Actual commit JSON:", latest)
print(raw_log["text"])

verify("update", "delta", spark.read.format("delta").load(DELTA_ROOT), inventory)
spark.stop()


### Hudi update

Read the operation, run it once, then inspect the actual metadata in the next cell.


In [ ]:
spark = open_spark("hudi")
assert progress["hudi"] == "insert", "Run cells once in order"
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
updated_order = [row for row in expected["update"] if row["order_id"] == UPDATE_KEY]
(spark.createDataFrame(updated_order, SCHEMA).write.format("hudi").options(**HUDI_OPTIONS)
      .option("hoodie.datasource.write.operation", "upsert").mode("append").save(HUDI_ROOT))


In [ ]:
metadata_reads.clear()
inventory = s3_files(spark, HUDI_ROOT)
main = [item for item in inventory if "/.hoodie/metadata/" not in item["path"]]
auxiliary = [item for item in inventory if "/.hoodie/metadata/" in item["path"]]
timeline = [item["path"] for item in main if "/.hoodie/timeline/" in item["path"]]
display(JSON({"main_timeline": timeline, "business_log_objects": sum(".log." in item["path"] for item in main),
              "auxiliary_metadata_log_objects": sum(".log." in item["path"] for item in auxiliary)}))
display(JSON(read_metadata(spark, HUDI_ROOT + "/.hoodie/hoodie.properties")))
completed = [path for path in timeline if path.endswith(".commit")]
assert completed, "No completed business commit found"
display(JSON(read_metadata(spark, max(completed))))

verify("update", "hudi", spark.read.format("hudi").load(HUDI_ROOT), inventory)
spark.stop()


## 10. Delete deterministic targets

Delete the configured selected keys, starting at sample rank 2 and wrapping if necessary. Defaults remove one order: 5 to 4. Logical deletion does not imply that every old physical file disappears. With zero deletions, skip the write and inspect the unchanged metadata.

Each format gets its own newly configured Spark context. The writes and metadata queries are directly below.


### Iceberg delete

Read the operation, run it once, then inspect the actual metadata in the next cell.


In [ ]:
spark = open_spark("iceberg")
assert progress["iceberg"] == "update", "Run cells once in order"
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
if DELETE_KEYS:
    spark.sql(f"DELETE FROM {ICEBERG_TABLE} WHERE order_id IN ({DELETE_SQL})")
else:
    print("No-op: zero deletes; no commit requested")


In [ ]:
metadata_reads.clear()
spark.table(ICEBERG_TABLE + ".snapshots").select("snapshot_id", "operation", "manifest_list").show(truncate=False)
spark.table(ICEBERG_TABLE + ".manifests").show(truncate=False)
spark.table(ICEBERG_TABLE + ".files").select("file_path", "record_count").show(truncate=False)
inventory = s3_files(spark, ICEBERG_ROOT)
metadata_paths = [item["path"] for item in inventory if item["path"].endswith(".metadata.json")]
latest = max(metadata_paths, key=lambda path: int(re.search(r"v(\d+)\.metadata\.json$", path).group(1)))
raw_json = read_metadata(spark, latest)
display(JSON(json.loads(raw_json["text"])))
manifest_list = json.loads(raw_json["text"])["snapshots"][-1]["manifest-list"]
display(JSON(read_metadata(spark, manifest_list)))

verify("delete", "iceberg", spark.table(ICEBERG_TABLE), inventory)
spark.stop()


### Delta delete

Read the operation, run it once, then inspect the actual metadata in the next cell.


In [ ]:
spark = open_spark("delta")
assert progress["delta"] == "update", "Run cells once in order"
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
if DELETE_KEYS:
    spark.sql(f"DELETE FROM {DELTA_TABLE} WHERE order_id IN ({DELETE_SQL})")
else:
    print("No-op: zero deletes; no commit requested")


In [ ]:
metadata_reads.clear()
spark.sql("DESCRIBE HISTORY " + DELTA_TABLE).select("version", "operation", "operationMetrics").show(truncate=False)
inventory = s3_files(spark, DELTA_ROOT)
log_paths = [item["path"] for item in inventory if "/_delta_log/" in item["path"] and item["path"].endswith(".json")]
latest = max(log_paths)
raw_log = read_metadata(spark, latest)
print("Actual commit JSON:", latest)
print(raw_log["text"])

verify("delete", "delta", spark.read.format("delta").load(DELTA_ROOT), inventory)
spark.stop()


### Hudi delete

Read the operation, run it once, then inspect the actual metadata in the next cell.


In [ ]:
spark = open_spark("hudi")
assert progress["hudi"] == "update", "Run cells once in order"
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
if DELETE_KEYS:
    deleted_orders = [row for row in expected["update"] if row["order_id"] in DELETE_KEYS]
    (spark.createDataFrame(deleted_orders, SCHEMA).write.format("hudi").options(**HUDI_OPTIONS)
          .option("hoodie.datasource.write.operation", "delete").mode("append").save(HUDI_ROOT))
else:
    print("No-op: zero deletes; no commit requested")


In [ ]:
metadata_reads.clear()
inventory = s3_files(spark, HUDI_ROOT)
main = [item for item in inventory if "/.hoodie/metadata/" not in item["path"]]
auxiliary = [item for item in inventory if "/.hoodie/metadata/" in item["path"]]
timeline = [item["path"] for item in main if "/.hoodie/timeline/" in item["path"]]
display(JSON({"main_timeline": timeline, "business_log_objects": sum(".log." in item["path"] for item in main),
              "auxiliary_metadata_log_objects": sum(".log." in item["path"] for item in auxiliary)}))
display(JSON(read_metadata(spark, HUDI_ROOT + "/.hoodie/hoodie.properties")))
completed = [path for path in timeline if path.endswith(".commit")]
assert completed, "No completed business commit found"
display(JSON(read_metadata(spark, max(completed))))

verify("delete", "hudi", spark.read.format("hudi").load(HUDI_ROOT), inventory)
spark.stop()


## 11. Verify and retain this manual run

All twelve format/stage comparisons use independent expected values. Each stage's local JSON includes the business rows, object inventory and metadata actually read. The executed notebook is saved separately under ignored `evidence/olist/manual/executions/` when using `make -f Makefile.olist manual-execute`.

The explorer continues to read **automated evidence snapshots**, not these manual tables or the live S3 data. Manual artifacts are intentionally outside its automated schema and do not change its current selection. This notebook has no cleanup/reset command; leave the isolated prefix intact for inspection.

References: [Iceberg Spark configuration](https://iceberg.apache.org/docs/1.11.0/spark-configuration/), [Delta quick start](https://docs.delta.io/quick-start/), [Hudi 1.2.0 Spark setup](https://hudi.apache.org/docs/1.2.0/quick-start-guide/), and [Olist data license](../../DATA_LICENSE.md).


In [ ]:
assert len(checks) == 12 and all(item["status"] == "PASS" for item in checks)
assert len(session_checks) == 12
assert all(stage == "delete" for stage in progress.values())
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest() == SOURCE_SHA
report = {"status": "PASS", "run_id": RUN_ID, "configuration": CONFIG,
          "expected_counts": [len(expected[stage]) for stage in ["create", "insert", "update", "delete"]],
          "source_rows": SOURCE_ROWS, "source_sha256": SOURCE_SHA,
          "storage_prefix": RUN_PREFIX, "evidence_directory": str(EVIDENCE),
          "timestamp_convention": "UTC for this lab; original source timezone unverified",
          "update": "Synthetic one-day estimated-delivery shift for the first initial order",
          "checks": checks, "separate_spark_sessions": session_checks}
(EVIDENCE / "verification.json").write_text(json.dumps(report, indent=2))
display(JSON(report))
